# 2.5 — Score a checkpoint on the manually-labeled honest set

Fast alternative to running the full labeling pass in `3-...ipynb` just to read the
honest accuracy at the bottom. This predicts **only** the ~150 hand-labeled `unknown`
events and runs the same comparison, so you can sweep checkpoints (e.g. each
`attempt4-...-epoch-N`) in seconds instead of ~23 min.

Point `model_dir` at a checkpoint and Run All.

## Setup

In [1]:
import os
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import plotly.express as px

/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = '../data'
MODELS_DIR = '../models'

# Point this at any checkpoint to score it on the honest manual set.
model_dir = f'{MODELS_DIR}/attempt7-t5-leave-keywords-in-epoch-2/hf_transformer_model'

MANUAL_DIR = f'{DATA_DIR}/manual_labelled_data'
INPUT_CSV  = f'{DATA_DIR}/labeled.csv'                # source of clean_notes for the manual events
TRAIN_CSV  = f'{DATA_DIR}/labeled_balanced_20.csv'   # only for label fallback

## Acceptance table (lenient scoring)

Some classes are near-synonyms (`climate`/`environment`) or routinely interchangeable
for a human annotator. Alongside **strict** accuracy (exact match), score a **lenient**
accuracy where a prediction counts as correct if it shares an acceptance group with the
gold label. Edit `ACCEPTANCE_GROUPS` to tune what's considered interchangeable.

In [3]:
# --- Acceptance table -------------------------------------------------------
# Near-synonym classes that may be scored interchangeably. A prediction is
# "accepted" if pred matches any gold label (primary or a manual_label_alt) or
# shares a group below with one. Groups may overlap; acceptance checks shared
# membership only, so it never chains (blm<->discrimination and
# blm<->unjust law enforcement both pass, but discrimination<->unjust law
# enforcement does not). Comment a line out to tighten.
ACCEPTANCE_GROUPS = [
    {'climate', 'environment'},                                  # ecology / green
    {'discrimination', 'women rights', 'lgbtq', 'blm'},          # equality & identity
    {'unjust law enforcement', 'blm'},                           # policing / state violence
    {'immigration', 'discrimination'},                           # anti-migrant racism
    {'farmers', 'labor rights'},                                 # agrarian livelihoods
    {'public services', 'health care', 'education', 'housing'},  # public-sector provision
]

def gold_labels(row):
    """All hand-assigned valid labels for an event: primary + any '|'-joined alts."""
    g = {row['manual_label']}
    alt = row.get('manual_label_alt')
    if isinstance(alt, str) and alt.strip():
        g |= {x.strip() for x in alt.split('|') if x.strip()}
    return g

def accepts(golds, pred):
    """True if pred matches any gold label or shares an acceptance group with one."""
    return any(g == pred or any(g in s and pred in s for s in ACCEPTANCE_GROUPS) for g in golds)

## Load the checkpoint

In [4]:
print(f"--- Loading {model_dir} ---")
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# CPU on purpose: only ~80 events, so it's a few seconds and skips the slow GPU
# kernel warm-up — and leaves the GPU free for a training run.
load_device = torch.device("cpu")
loaded_model.to(load_device).eval()
print(f"Using device: {load_device}")

--- Loading ../models/attempt7-t5-leave-keywords-in-epoch-2/hf_transformer_model ---


Loading weights: 100%|██████████| 261/261 [00:00<00:00, 10447.83it/s]

Using device: cpu


In [5]:
# Class names from the model config; fall back to the training CSV for old
# models saved with generic LABEL_0..N names (mirrors notebook 3).
id2label = {int(k): v for k, v in loaded_model.config.id2label.items()}
if all(str(v).startswith("LABEL_") for v in id2label.values()):
    _t = pd.read_csv(TRAIN_CSV)
    _t = _t[~_t['class'].isin(['NoN', 'unknown'])]
    id2label = {i: name for i, name in enumerate(sorted(_t['class'].unique()))}
assert len(id2label) == loaded_model.config.num_labels

## Predict the manual events only

The manual CSVs carry `event_id_cnty` but not `clean_notes`, so pull the text from
`labeled.csv` by id, then run the model on just those rows.

In [6]:
ids = set(pd.read_csv(f'{MANUAL_DIR}/{'random_unknown_labeled.csv'}', usecols=['event_id_cnty'])['event_id_cnty'])

src = pd.read_csv(INPUT_CSV, usecols=['event_id_cnty', 'clean_notes'], low_memory=False)
eval_df = src[src['event_id_cnty'].isin(ids)].copy()
eval_df['clean_notes'] = eval_df['clean_notes'].fillna('').astype(str)
print(f"manual events: {len(ids)} | matched in labeled.csv: {len(eval_df)}")

texts = eval_df['clean_notes'].tolist()
preds, BATCH = [], 64
for i in range(0, len(texts), BATCH):
    batch = loaded_tokenizer(texts[i:i+BATCH], return_tensors='pt',
                             padding='longest', truncation=True, max_length=128)
    batch = {k: v.to(load_device) for k, v in batch.items()}
    with torch.inference_mode():
        logits = loaded_model(**batch).logits
    preds.extend(logits.argmax(dim=1).cpu().tolist())

eval_df['predicted_class'] = [id2label[p] for p in preds]
this_model = eval_df[['event_id_cnty', 'predicted_class']]

manual events: 200 | matched in labeled.csv: 200


## Honest comparison (same as the end of notebook 3)

In [7]:
def load_eval(path):
    d = pd.read_csv(path).drop(columns=['predicted_class_model'], errors='ignore')  # injected live below
    d = d.merge(this_model, on='event_id_cnty', how='left').rename(
        columns={'predicted_class': 'predicted_class_model'})
    d['model_correct'] = d['predicted_class_model'] == d['manual_label']
    d['students_correct'] = d['predicted_class_students'] == d['manual_label']
    # lenient: near-synonym classes + per-event multi-labels count as correct
    golds = d.apply(gold_labels, axis=1)
    d['model_accept'] = [accepts(g, p) for g, p in zip(golds, d['predicted_class_model'])]
    d['students_accept'] = [accepts(g, p) for g, p in zip(golds, d['predicted_class_students'])]
    d['agree'] = d['predicted_class_model'] == d['predicted_class_students']
    return d

rand = load_eval(f'{MANUAL_DIR}/random_unknown_labeled.csv')      # random unknown events (honest)
assert rand['predicted_class_model'].notna().all(), 'some events missing from labeled.csv'
print(f"random (honest): {len(rand)}")
print(f"strict  -> this model {rand['model_correct'].mean():.0%} | students {rand['students_correct'].mean():.0%}")
print(f"lenient -> this model {rand['model_accept'].mean():.0%} | students {rand['students_accept'].mean():.0%}")

random (honest): 200
strict  -> this model 55% | students 55%
lenient -> this model 66% | students 69%


In [8]:
n = len(rand)
ci = lambda p: 1.96 * (p * (1 - p) / n) ** 0.5  # normal-approx 95% CI (small n!)

head = pd.DataFrame([
    {'source': 'This model', 'scoring': 'strict',  'accuracy': rand['model_correct'].mean()},
    {'source': 'This model', 'scoring': 'lenient', 'accuracy': rand['model_accept'].mean()},
    {'source': 'Students',   'scoring': 'strict',  'accuracy': rand['students_correct'].mean()},
    {'source': 'Students',   'scoring': 'lenient', 'accuracy': rand['students_accept'].mean()},
])
head['ci'] = head['accuracy'].map(ci)
fig = px.bar(head, x='source', y='accuracy', color='scoring', barmode='group',
             text=head['accuracy'].round(3), error_y=head['ci'],
             labels={'source': '', 'accuracy': 'Accuracy on random unknown'},
             color_discrete_map={'strict': '#636EFA', 'lenient': '#00CC96'})
fig.update_layout(title={'text': f'Honest accuracy: strict vs lenient (n={n}, 95% CI)', 'x': 0.5},
                  plot_bgcolor='white', width=680, height=440, yaxis={'range': [0, 1]})
fig.show()

In [9]:
# Per-event labels: manual (truth) + any alt golds, our model, students, whether the
# model's label is accepted under ACCEPTANCE_GROUPS, and clean_notes (by event id).
cols = {'manual_label': 'manual', 'manual_label_alt': 'alt', 'predicted_class_model': 'our model',
        'predicted_class_students': 'students', 'model_accept': 'accepted', 'clean_notes': 'text'}

def table(df):
    return (df.merge(eval_df[['event_id_cnty', 'clean_notes']], on='event_id_cnty', how='left')
              [list(cols)].rename(columns=cols))


wrong = rand[~(rand['model_accept'] & rand['students_accept'])]   # at least one of them wrong
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(f"At least one wrong (lenient): {len(wrong)}/{len(rand)}")
    display(table(wrong))

At least one wrong (lenient): 87/200


,manual,alt,our model,students,accepted,text
0,labor rights,NaN,public services,pandemic,False,striking stv journalist represented national union journalist nuj held picket outside scottish parliament edinburgh scotland row pay rise cost living crisis event part national day action
1,labor rights,NaN,climate,farmers,False,hundred unionized employee alro aluminium plant protested slatina complaining high price energy led reduction production capacity demanding permission company negotiate energy delivery directly energy producer
2,climate,NaN,public services,climate,False,extinction rebellion activist gathered outside stock exchange milano municipio 1 staged flash mob protest climate crisis demand finance tackle climate emergency
3,policies & politics,NaN,public services,housing,False,relative victim member mayor executive unrolled banner balcony town hall palazzo pretorio palermo palermo sicilia demand justice regarding death italian un aid worker found dead apartment san vincente del caguan colombia july 2020
4,climate,NaN,pandemic,environment,False,around 150 people mostly fff member cycled demonstrated halle saale call climate protection occasion co27 egypt cycled universitatsplatz passage 13 neustadt
5,environment,NaN,pandemic,environment,False,10 member rouvikonas threw flyer premise construction company northern athens protest construction landfill planned corfu
6,policies & politics,NaN,policies & politics,lgbtq,True,pegida staged protest dresden sachsen motto foot quergida fascism people staged counter protest several people held sit nearby protest motif specified
7,labor rights,NaN,labor rights,policies & politics,True,employee macedonian information agency protested skopje demanding salary national news agency increased parliament consider request
8,labor rights,farmers,climate,pandemic,False,striking fisher gathered port gallipoli lecce puglia protest increase oil gas price denounce impact category
9,labor rights,NaN,labor rights,public services,True,taxi driver representing app driver courier union gathered outside uber office manchester part 24 hour strike protest poor pay condition unfair dismissal driver demanded rate set 2 gbp per mile part series co ordinated strike member union across several city country attendance representative unite union


In [10]:
# Per-event labels: manual (truth) + any alt golds, our model, students, whether the
# model's label is accepted under ACCEPTANCE_GROUPS, and clean_notes (by event id).
cols = {'manual_label': 'manual', 'manual_label_alt': 'alt', 'predicted_class_model': 'our model',
        'predicted_class_students': 'students', 'model_accept': 'accepted', 'clean_notes': 'text'}

def table(df):
    return (df.merge(eval_df[['event_id_cnty', 'clean_notes']], on='event_id_cnty', how='left')
              [list(cols)].rename(columns=cols))


with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(table(rand))

,manual,alt,our model,students,accepted,text
0,labor rights,NaN,labor rights,labor rights,True,morning call fnv around 100 striking iff worker staged picket outside factory gate tilburg noord brabant demand 7 wage increase senior scheme better condition following initial 24 hour strike earlier week
1,policies & politics,NaN,policies & politics,policies & politics,True,people demonstrated paris call bank government drop public debt african country
2,labor rights,NaN,public services,pandemic,False,striking stv journalist represented national union journalist nuj held picket outside scottish parliament edinburgh scotland row pay rise cost living crisis event part national day action
3,labor rights,NaN,climate,farmers,False,hundred unionized employee alro aluminium plant protested slatina complaining high price energy led reduction production capacity demanding permission company negotiate energy delivery directly energy producer
4,climate,NaN,public services,climate,False,extinction rebellion activist gathered outside stock exchange milano municipio 1 staged flash mob protest climate crisis demand finance tackle climate emergency
5,climate,NaN,environment,climate,True,throughout afternoon 11 member last generation gathered konrad adenauer bridge mannheim ludwigshafen mannheim climate protection 5 people stuck road direction mannheim 4 activist side road another action took place feeding lane square q6 q7
6,policies & politics,NaN,public services,housing,False,relative victim member mayor executive unrolled banner balcony town hall palazzo pretorio palermo palermo sicilia demand justice regarding death italian un aid worker found dead apartment san vincente del caguan colombia july 2020
7,policies & politics,NaN,policies & politics,policies & politics,True,three activist gathered protest izola planned purchase local dock military
8,climate,NaN,pandemic,environment,False,around 150 people mostly fff member cycled demonstrated halle saale call climate protection occasion co27 egypt cycled universitatsplatz passage 13 neustadt
9,labor rights,NaN,labor rights,labor rights,True,striking amazon worker represented gmb picketed company fulfillment center coventry dispute pay working condition
